# Vilex — Kaggle Stage 5 only (OmniVoice TTS, Vietnamese)

**Repo:** https://github.com/thaiphu05/Vilex @ `dev-thai`  
**Input:** Stage 1–4b JSONs (`data/vi_tt_bc`, uploaded as a Kaggle Dataset) + a `voice_clone/` pool  
**Output:** `data/vi_audio/**/varNN/dialogues/dialogue.wav` (stereo, ch0=assistant ch1=user) + `meta.json` (+ `alignment_user.json`/`alignment_assistant.json`)

Stage 5 reads `config.yaml`; this notebook overrides it through `VILEX_*` env vars (dotted key with `__`, e.g. `VILEX_STAGE5_TTS__DEVICE`). See `docs/CONFIGURATION.md` and `docs/stage5-tts.md`.

## Kaggle settings (do once)
- `Settings → Internet ON` (pip installs + NLTK punkt + HF weight download)
- `Settings → Accelerator → GPU T4 x2`
- `Add Input`: (1) the Stage 1–4b dialogues (`data/vi_tt_bc`) and (2) the `voice_clone` pool (each `*.wav` needs a sidecar `*.txt`)
- No `Secrets` needed — Stage 5 makes no Gemini calls, and this notebook uses a single environment

In [ ]:
# Cell 1 — Knobs (edit these)
BRANCH = "dev-thai"
REPO = "https://github.com/thaiphu05/Vilex.git"

BC_ROOT = "data/vi_tt_bc"            # empty/None -> auto-discover under /kaggle/input
VOICE_POOL = "/kaggle/input/voice-clone"  # auto-discovered too; needs >=2 wav+txt

DEVICE = "cuda"                     # fall back to "cpu" on OOM
NUM_VARIANTS = 1                     # variants per dialogue (linear cost)
MAX_DIALOGUES = 1                    # 1 = smoke test; 0 = all
SAVE_ALIGN_JSON = True               # write alignment_user/assistant.json (word timestamps)
RENDER_TAGS = True                   # keep [laughter]/[sigh]/... supported tags

In [ ]:
# Cell 2 — Clone code (data comes from Kaggle Datasets)
!git clone -b $BRANCH $REPO
%cd Vilex
# config.yaml is gitignored; copy the example so Stage 5 has a deterministic base
# (VILEX_* env overrides still win).
!cp config_example.yaml config.yaml
!git branch --show-current && git log --oneline -3 && ls -lh

In [ ]:
# Cell 3 — Install Stage 5 deps (lean OmniVoice/Vietnamese path)
# Skips nemo-text-processing (Chatterbox/English only) to keep the Kaggle env small.
# transformers>=4.56 backs the Qwen3 forced aligner (replaced whisperx).
!pip install -q torchaudio nltk silero-vad 'transformers>=4.56' pyyaml pyloudnorm soundfile librosa
!pip install -q git+https://github.com/k2-fsa/OmniVoice.git
# OmniVoice may pull an older transformers; re-assert the aligner requirement.
!pip install -q 'transformers>=4.56'

In [ ]:
# Cell 4 — Verify imports + CUDA
!python -c 'import torch, torchaudio, silero_vad; from transformers import AutoProcessor, AutoModelForTokenClassification; from omnivoice import OmniVoice; print(torch.__version__, torchaudio.__version__, torch.cuda.is_available())'

In [ ]:
# Cell 5 — Discover the input dialogues + voice-clone pool
import os, glob

def find_bc_root(root):
    for cand in [root, "data/vi_tt_bc"]:
        if cand and glob.glob(os.path.join(cand, "**", "text_dialogue_*", "*", "*.json"), recursive=True):
            return cand
    for cand in sorted(glob.glob("/kaggle/input/*")):
        if glob.glob(os.path.join(cand, "**", "text_dialogue_*", "*", "*.json"), recursive=True):
            return cand
    return None

def find_voice_pool(root):
    for cand in [root, "/kaggle/input/voice-clone", "/kaggle/input/voice_clone",
                 "voice_clone", "/kaggle/working/voice_clone"]:
        if cand and os.path.isdir(cand) and glob.glob(os.path.join(cand, "*.wav")):
            return cand
    return None

BC_ROOT = find_bc_root(BC_ROOT)
VOICE_POOL = find_voice_pool(VOICE_POOL)

if not BC_ROOT:
    raise SystemExit("No Stage 1-4b dialogues found. Add Input with data/vi_tt_bc (text_dialogue_*/*/*.json).")
if not VOICE_POOL:
    raise SystemExit("No voice_clone pool found (need >=2 .wav with sidecar .txt).")

n_json = glob.glob(os.path.join(BC_ROOT, "**", "text_dialogue_*", "*", "*.json"), recursive=True)
wavs = sorted(glob.glob(os.path.join(VOICE_POOL, "*.wav")))
missing = [w for w in wavs if not os.path.isfile(os.path.splitext(w)[0] + ".txt")]
print(f"BC_ROOT={BC_ROOT}  ({len(n_json)} dialogues)")
print(f"VOICE_POOL={VOICE_POOL}  ({len(wavs)} wavs, {len(missing)} missing .txt)")
if len(wavs) < 2:
    raise SystemExit("voice pool needs >=2 wavs with sidecar .txt")

In [ ]:
# Cell 6 — Override config.yaml via VILEX_* env vars
import os, json

def set_vi(overrides):
    for key, val in overrides.items():
        if isinstance(val, bool):
            sval = "true" if val else "false"
        elif isinstance(val, (list, dict)):
            sval = json.dumps(val, ensure_ascii=False)
        else:
            sval = str(val)
        os.environ["VILEX_" + key.upper().replace(".", "__")] = sval

set_vi({
    "paths.bc_root": BC_ROOT,                 # recursive glob: any depth under it
    "paths.audio_root": "data/vi_audio",
    "paths.voice_clone_pool": VOICE_POOL,
    "stage5_tts.backend": "omnivoice",
    "stage5_tts.language": "vi",
    "stage5_tts.device": DEVICE,
    "stage5_tts.num_variants": NUM_VARIANTS,
    "stage5_tts.max_dialogues": MAX_DIALOGUES,
    "stage5_tts.tags.render": RENDER_TAGS,
    "stage5_tts.audio.save_align_json": SAVE_ALIGN_JSON,
})
print(f"device={DEVICE} variants={NUM_VARIANTS} max_dialogues={MAX_DIALOGUES} align_json={SAVE_ALIGN_JSON} tags={RENDER_TAGS}")

In [ ]:
# Cell 7 — Stage 5: OmniVoice render
!python tts_render/convert_spoken.py

In [ ]:
# Cell 8 — If CUDA OOM, rerun on CPU (resume skips finished variants)
import os
os.environ["VILEX_STAGE5_TTS__DEVICE"] = "cpu"
!python tts_render/convert_spoken.py

In [ ]:
# Cell 9 — Preview + export
!find data/vi_audio -type f 2>/dev/null | head -30
try:
    from IPython.display import Audio, display
    import glob as _g
    wavs = _g.glob("data/vi_audio/**/dialogue.wav", recursive=True)
    if wavs:
        print(f"Preview: {wavs[0]}")
        display(Audio(wavs[0]))
    else:
        print("No dialogue.wav yet - check logs above")
except Exception as e:
    print(e)
!zip -qr /kaggle/working/vi_audio.zip data/vi_audio
!ls -lh /kaggle/working/vi_audio.zip 2>/dev/null

## Notes
- **Full run:** set `MAX_DIALOGUES = 0` (and raise `NUM_VARIANTS`) in Cell 1. Rendering runs at roughly real time, so a full pass is slow.
- **Resume:** a variant whose `dialogue.wav` + `meta.json` exist (plus `alignment_*.json` when `SAVE_ALIGN_JSON`) is skipped, so re-running after an OOM only fills the gaps.
- **Alignment JSON:** with `SAVE_ALIGN_JSON = True`, each `varNN/` also gets `alignment_user.json` / `alignment_assistant.json` (absolute word timestamps per speaker).
- **Override anything without editing files:** `VILEX_STAGE5_TTS__DEVICE=cuda`, `VILEX_STAGE5_TTS__MAX_DIALOGUES=0`, ... — see `docs/CONFIGURATION.md`.
- **Output layout:** `data/vi_audio/text_dialogue_<ds>/<split>/<id>/varNN/dialogues/dialogue.wav` (stereo, ch0=assistant ch1=user) + `meta.json`.
- **Upload Stage 1–4b output:** locally `zip -r vilex-stage4b-dialogues.zip data/vi_tt_bc` → `Kaggle → Datasets → New` (`data/` is gitignored).